# 01 - Exploración de Datos: Base de Datos de Arritmias MIT-BIH

**Fase del Pipeline DSR: Comprensión del Problema y Adquisición de Datos**

Este notebook explora la base de datos de arritmias MIT-BIH antes del preprocesamiento.
Examinamos:
- Los 48 registros y sus categorías de patología
- Agrupación de símbolos de anotación AAMI (Normal vs Anómalo)
- Balance de clases en el conjunto de datos
- Visualización de señales ECG crudas

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from ecg_anomaly.config import SystemConfig
from ecg_anomaly.seeding import set_global_seed
from ecg_anomaly.data.registry import RecordRegistry, AAMI_NORMAL, AAMI_ANOMALOUS
from ecg_anomaly.data.loader import MITBIHLoader
from ecg_anomaly.visualization.labels import LABELS_ES

sns.set_style("whitegrid")
config = SystemConfig.from_yaml("../config/default.yaml")
set_global_seed(config.random_seed)
registry = RecordRegistry()

## 1. Catálogo de Registros

MIT-BIH contiene 48 registros. Excluimos 4 registros con marcapasos (102, 104, 107, 217)
siguiendo las recomendaciones AAMI, quedando 44 registros utilizables.

In [ ]:
print(f"Total de registros en MIT-BIH: {len(RecordRegistry.ALL_RECORDS)}")
print(f"Excluidos (marcapasos): {sorted(config.excluded_records)}")

valid_records = registry.get_valid_records(set(config.excluded_records))
print(f"Registros validos para el analisis: {len(valid_records)}")
print(f"IDs de registro: {valid_records}")

## 2. Categorías de Registros por Patología

Los registros se agrupan por su tipo de arritmia predominante.

In [ ]:
categories = registry.get_categories()
print("Categorias de registros:\n")
for cat in categories:
    records = registry.get_records_by_category(cat)
    print(f"  {cat}: {len(records)} registros -> {records}")

# Bar chart of category sizes
cat_sizes = {cat: len(registry.get_records_by_category(cat)) for cat in categories}
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(cat_sizes.keys()), list(cat_sizes.values()), color="#2E86AB")
ax.set_xlabel("Numero de registros")
ax.set_title(LABELS_ES["MIT-BIH Records by Pathology Category"])
for i, v in enumerate(cat_sizes.values()):
    ax.text(v + 0.2, i, str(v), va="center")
plt.tight_layout()
plt.show()

## 3. Agrupación de Símbolos AAMI

El estándar AAMI agrupa los símbolos de anotación de MIT-BIH en clases clínicamente significativas.
Para la detección binaria de anomalías usamos:
- **Normal (0):** N, L, R, e, j
- **Anómalo (1):** A, a, J, S, V, E, F, /, f, Q

In [ ]:
print("Simbolos normales AAMI:", sorted(AAMI_NORMAL))
print("Simbolos anomalos AAMI:", sorted(AAMI_ANOMALOUS))
print()
print("Ejemplos de clasificacion:")
for sym in ["N", "L", "R", "V", "A", "F", "/", "+", "~"]:
    label = RecordRegistry.classify_symbol(sym)
    desc = {0: "Normal", 1: "Anomalo", -1: "No es un latido (ignorado)"}[label]
    print(f"  '{sym}' -> {label} ({desc})")

## 4. Cargar Dataset y Examinar el Balance de Clases

In [ ]:
loader = MITBIHLoader(config)
dataset = loader.load(config.dataset_path)

print(f"Registros cargados: {len(dataset.records)}")
print(f"Total de latidos: {dataset.total_beats:,}")
print(f"Latidos normales: {dataset.total_normal:,} ({dataset.total_normal/dataset.total_beats*100:.1f}%)")
print(f"Latidos anomalos: {dataset.total_anomalous:,} ({dataset.total_anomalous/dataset.total_beats*100:.1f}%)")

In [ ]:
# Global class balance pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
axes[0].pie(
    [dataset.total_normal, dataset.total_anomalous],
    labels=["Normal", "Anomalo"],
    colors=["#2E86AB", "#E74C3C"],
    autopct="%1.1f%%",
    startangle=90,
)
axes[0].set_title(LABELS_ES["AAMI Binary Class Distribution"])

# Symbol distribution across all records
all_symbols = []
for rec in dataset.records:
    all_symbols.extend(rec.symbols.tolist())

symbol_counts = Counter(all_symbols)
symbols_sorted = sorted(symbol_counts.items(), key=lambda x: -x[1])
syms, counts = zip(*symbols_sorted)
colors = ["#2E86AB" if s in AAMI_NORMAL else "#E74C3C" for s in syms]

axes[1].bar(syms, counts, color=colors)
axes[1].set_ylabel("Cantidad")
axes[1].set_title(LABELS_ES["Annotation Symbol Distribution"])
axes[1].set_yscale("log")
axes[1].set_xlabel("Simbolo (azul=Normal, rojo=Anomalo)")

plt.tight_layout()
plt.show()

## 5. Balance de Clases por Registro

Algunos registros son predominantemente normales, otros contienen muchas arritmias.

In [ ]:
record_stats = []
for rec in dataset.records:
    n_normal = int(np.sum(rec.binary_labels == 0))
    n_anomalous = int(np.sum(rec.binary_labels == 1))
    record_stats.append((rec.record_id, n_normal, n_anomalous))

record_stats.sort(key=lambda x: x[2] / (x[1] + x[2]), reverse=True)

fig, ax = plt.subplots(figsize=(14, 6))
ids = [r[0] for r in record_stats]
normals = [r[1] for r in record_stats]
anomalous = [r[2] for r in record_stats]

ax.bar(ids, normals, label="Normal", color="#2E86AB")
ax.bar(ids, anomalous, bottom=normals, label="Anomalo", color="#E74C3C")
ax.set_xlabel("ID de registro")
ax.set_ylabel("Numero de latidos")
ax.set_title(LABELS_ES["Per-Record Class Distribution (sorted by anomaly ratio)"])
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

## 6. Visualización de Señal ECG Cruda

Visualiza algunos segundos de señal cruda de registros seleccionados.

In [ ]:
# Pick 3 records: one normal-dominant, one with many anomalies, one mixed
sample_ids = ["100", "233", "208"]
sample_records = [r for r in dataset.records if r.record_id in sample_ids]

fig, axes = plt.subplots(len(sample_records), 1, figsize=(14, 3 * len(sample_records)))
if len(sample_records) == 1:
    axes = [axes]

for ax, rec in zip(axes, sample_records):
    duration = 5  # seconds
    n_samples = duration * rec.sampling_rate
    t = np.arange(min(n_samples, len(rec.signal))) / rec.sampling_rate
    sig = rec.signal[:len(t)]

    ax.plot(t, sig, linewidth=0.5, color="#2E86AB")

    # Mark R-peaks in this window
    peaks_in_window = rec.r_peak_positions[rec.r_peak_positions < len(t)]
    ax.plot(peaks_in_window / rec.sampling_rate, sig[peaks_in_window], "rv", markersize=6)

    n_anom = int(np.sum(rec.binary_labels == 1))
    ax.set_title(f"Registro {rec.record_id} ({len(rec.binary_labels)} latidos, {n_anom} anomalos)")
    ax.set_ylabel("mV")

axes[-1].set_xlabel("Tiempo (s)")
plt.tight_layout()
plt.show()

## Resumen

Observaciones clave:
- El dataset tiene **desequilibrio de clases**: los latidos normales superan significativamente a los anómalos.
- El símbolo dominante es 'N' (sinusal normal); 'V' (PVC) y 'A' (APC) son las anomalías más comunes.
- Algunos registros contienen casi exclusivamente latidos normales, mientras que otros tienen muchas arritmias.
- Este desequilibrio es un desafío clave para la detección de anomalías no supervisada.

**Siguiente:** `02_preprocessing.ipynb` -- filtrar, segmentar y normalizar las señales.